# Chapter 15 Computational Lab
## Monte Carlo Methods: Simulation, Error and Variance Reduction

This notebook accompanies Chapter 15 of *Probability Theory with Python and AI*.

Monte Carlo methods turn expectation into computation. The basic pattern is

$$
\boxed{
\theta=\mathbb E[g(X)]
\quad\longrightarrow\quad
\widehat\theta_n
=
\frac1n
\sum_{i=1}^n
g(X_i).
}
$$

The law of large numbers explains consistency. The central limit theorem and
studentization quantify random sampling error. Variance-reduction methods aim to
make the random payoff less variable without changing its target mean.

### Learning goals

By the end of the lab you should be able to:

1. define the plain Monte Carlo estimator;
2. distinguish MSE, RMSE and standard error;
3. derive the $n^{-1/2}$ square-root law;
4. use the strong law for Monte Carlo consistency;
5. compute a rigorous Chebyshev finite-sample guarantee;
6. use the Monte Carlo CLT and studentized asymptotic intervals;
7. distinguish asymptotic coverage from exact finite-sample coverage;
8. generate samples by generalized inverse transformation;
9. implement rejection sampling and understand why its acceptance probability is $1/M$;
10. formulate numerical integration as expectation;
11. distinguish sampling error from deterministic numerical bias;
12. use the bias--variance decomposition;
13. implement and analyze antithetic variates;
14. derive the optimal one-dimensional control-variate coefficient;
15. use several control variates through a covariance matrix;
16. construct stratified estimators and compute their variances;
17. explain proportional stratification using the law of total variance;
18. derive and use Neyman allocation;
19. state importance sampling as a change-of-measure identity;
20. check the support condition before using importance weights;
21. estimate rare-event probabilities with importance sampling;
22. explain why rare events make plain Monte Carlo inefficient in relative error;
23. compare hit-or-miss and direct-integration estimators of $\pi$;
24. document random-number generation for reproducibility;
25. separate Monte Carlo error from model risk;
26. follow a mathematically defensible Monte Carlo workflow;
27. audit AI-generated Monte Carlo claims.

> **Core principle.** A small Monte Carlo standard error means only that sampling noise under the chosen model is small. It does not validate the model itself.


## 0. Setup

The notebook uses direct formulas and NumPy simulation. Every variance-reduction method is tied back to the exact estimator and its probability law.


In [ ]:
from math import ceil
from statistics import NormalDist
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


STD_NORMAL = NormalDist()


def normal_cdf(x):
    return STD_NORMAL.cdf(x)


def normal_sf(x):
    return 0.5*math.erfc(x/math.sqrt(2))


def normal_quantile(p):
    return STD_NORMAL.inv_cdf(p)


def mc_summary(sample):
    sample = np.asarray(sample, dtype=float)
    n = sample.size

    estimate = float(np.mean(sample))
    if n >= 2:
        sample_sd = float(np.std(sample, ddof=1))
        se = sample_sd/math.sqrt(n)
    else:
        sample_sd = float("nan")
        se = float("nan")

    return {
        "n": n,
        "estimate": estimate,
        "sample_sd": sample_sd,
        "se": se,
    }


def chebyshev_sample_size(variance, epsilon, delta):
    return math.ceil(variance/(delta*epsilon*epsilon))


def asymptotic_ci(sample, alpha=0.05):
    summary = mc_summary(sample)
    z = normal_quantile(1-alpha/2)
    half = z*summary["se"]

    return (
        summary["estimate"]-half,
        summary["estimate"]+half,
        summary["estimate"],
        summary["se"],
    )


def rejection_beta21(number_of_proposals, seed=2026):
    rng = np.random.default_rng(seed)
    x = rng.random(number_of_proposals)
    u = rng.random(number_of_proposals)

    accepted = x[u <= x]

    return accepted


def direct_pi_payoff(u):
    u = np.asarray(u, dtype=float)
    return 4*np.sqrt(1-u*u)


def hit_miss_pi_payoff(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return 4*((u*u+v*v) <= 1).astype(float)


def control_optimal_coefficient(y, c):
    y = np.asarray(y, dtype=float)
    c = np.asarray(c, dtype=float)

    cov = np.cov(y, c, ddof=1)[0,1]
    var_c = np.var(c, ddof=1)

    return cov/var_c


def normal_tail_is_payoff(x, shift):
    # Target f=N(0,1), proposal q=N(shift,1).
    # f(x)/q(x)=exp(shift^2/2-shift*x).
    x = np.asarray(x, dtype=float)

    weight = np.exp(0.5*shift*shift-shift*x)
    return (x > 4.0)*weight


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))

    for line in latex_lines:
        display(Math(line))

    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Monte Carlo tools are ready."
    "</div>"
))


## 1. The basic Monte Carlo problem

Suppose

$$
\theta
=
\mathbb E[g(X)].
$$

Let $X_1,\ldots,X_n$ be independent copies of $X$ and define

$$
Y_i=g(X_i).
$$

The **plain Monte Carlo estimator** is

$$
\boxed{
\widehat\theta_n
=
\overline Y_n
=
\frac1n
\sum_{i=1}^nY_i.
}
$$


### Error measures

For an estimator $T$ of $\theta$,

$$
\operatorname{MSE}(T;\theta)
=
\mathbb E[(T-\theta)^2],
$$

$$
\operatorname{RMSE}(T;\theta)
=
\sqrt{
\operatorname{MSE}(T;\theta)
},
$$

and, when $T\in L^2$,

$$
\operatorname{SE}(T)
=
\sqrt{\operatorname{Var}(T)}.
$$

If $T$ is unbiased, then

$$
\boxed{
\operatorname{RMSE}(T;\theta)
=
\operatorname{SE}(T).
}
$$


### Mean and variance of plain Monte Carlo

If

$$
\mathbb E[Y]=\theta,
\qquad
\operatorname{Var}(Y)=\sigma^2,
$$

then

$$
\boxed{
\mathbb E[\widehat\theta_n]
=
\theta,
}
$$

and

$$
\boxed{
\operatorname{Var}(\widehat\theta_n)
=
\frac{\sigma^2}{n}.
}
$$

Therefore

$$
\boxed{
\operatorname{RMSE}(\widehat\theta_n;\theta)
=
\operatorname{SE}(\widehat\theta_n)
=
\frac{\sigma}{\sqrt n}.
}
$$


## 2. The Monte Carlo square-root law

Plain Monte Carlo standard error decays at rate

$$
n^{-1/2}.
$$

Reducing the standard error by a factor of $10$ requires approximately $100$ times as many independent samples.

This slow rate is a main reason for variance reduction.


In [ ]:
sqrt_sigma = widgets.FloatSlider(
    value=2.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="sigma",
)
sqrt_output = widgets.Output()


def update_square_root(*_):
    with sqrt_output:
        clear_output(wait=True)

        sigma = sqrt_sigma.value
        n = np.logspace(1, 6, 300)
        se = sigma/np.sqrt(n)

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.loglog(n, se)
        ax.set_xlabel("n")
        ax.set_ylabel("standard error")
        ax.set_title("Monte Carlo square-root law")
        plt.show()

        display(Math(
            r"\operatorname{SE}_{10^4}="
            + f"{sigma/100:.6f}"
        ))
        display(Math(
            r"\operatorname{SE}_{10^6}="
            + f"{sigma/1000:.6f}"
        ))


sqrt_sigma.observe(update_square_root, names="value")
display(widgets.VBox([sqrt_sigma, sqrt_output]))
update_square_root()


### One extra decimal place

If $\sigma=2$ and $n=10^4$,

$$
\operatorname{SE}(\widehat\theta_n)
=
0.02.
$$

To reduce this to $0.002$, plain Monte Carlo requires approximately

$$
\boxed{
n=10^6.
}
$$

One extra decimal place of Monte Carlo error typically costs about two extra decimal places in sample size.


## 3. Strong consistency from the law of large numbers

If $Y_1,Y_2,\ldots$ are i.i.d. and

$$
\mathbb E|Y_1|<\infty,
$$

then

$$
\boxed{
\widehat\theta_n
\longrightarrow
\theta
\quad
\text{almost surely}.
}
$$

In particular,

$$
\widehat\theta_n
\xrightarrow{P}
\theta.
$$

Consistency is asymptotic. It does not quantify the error of one finite run.


### Hit-or-miss $\pi$

Let

$$
Y_i
=
4\mathbf 1_{\{U_i^2+V_i^2\le1\}},
$$

with independent uniform points in the unit square.

Then

$$
\mathbb E[Y_i]
=
\pi,
$$

so

$$
\boxed{
\widehat\pi_n
=
\frac1n
\sum_{i=1}^nY_i
\longrightarrow
\pi
\quad
\text{a.s.}
}
$$


In [ ]:
pi_N = widgets.IntSlider(
    value=20000,
    min=500,
    max=100000,
    step=500,
    description="N",
)
pi_seed = widgets.IntSlider(
    value=2026,
    min=0,
    max=5000,
    description="seed",
)
pi_output = widgets.Output()


def update_pi_path(*_):
    with pi_output:
        clear_output(wait=True)

        N = pi_N.value
        seed = pi_seed.value

        rng = np.random.default_rng(seed)
        u = rng.random(N)
        v = rng.random(N)

        y = hit_miss_pi_payoff(u,v)
        running = np.cumsum(y)/np.arange(1,N+1)

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(np.arange(1,N+1), running)
        ax.axhline(math.pi, linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("pi estimate")
        ax.set_title("Strong-consistency picture for hit-or-miss pi")
        plt.show()

        display(Math(
            r"\widehat\pi_N="
            + f"{running[-1]:.6f}"
        ))


for control in (pi_N, pi_seed):
    control.observe(update_pi_path, names="value")

display(widgets.VBox([
    widgets.HBox([pi_N, pi_seed]),
    pi_output,
]))
update_pi_path()


## 4. Finite-sample bounds and central-limit error

If $Y\in L^2$ with variance $\sigma^2$, then Chebyshev gives, for every $\varepsilon>0$,

$$
\boxed{
P(
|\widehat\theta_n-\theta|
\ge
\varepsilon
)
\le
\frac{\sigma^2}{n\varepsilon^2}.
}
$$

If $0<\delta<1$, it is sufficient to take

$$
\boxed{
n
\ge
\frac{\sigma^2}{\delta\varepsilon^2}
}
$$

to guarantee

$$
P(
|\widehat\theta_n-\theta|
<
\varepsilon
)
\ge
1-\delta.
$$


In [ ]:
cheb_var = widgets.FloatSlider(
    value=4.0,
    min=0.1,
    max=25.0,
    step=0.1,
    description="variance",
)
cheb_eps = widgets.FloatSlider(
    value=0.05,
    min=0.005,
    max=0.2,
    step=0.005,
    description="epsilon",
)
cheb_delta = widgets.FloatSlider(
    value=0.01,
    min=0.005,
    max=0.2,
    step=0.005,
    description="delta",
)
cheb_output = widgets.Output()


def update_cheb(*_):
    with cheb_output:
        clear_output(wait=True)

        variance = cheb_var.value
        eps = cheb_eps.value
        delta = cheb_delta.value

        n = chebyshev_sample_size(
            variance,
            eps,
            delta,
        )

        display(Math(
            r"n_{\mathrm{sufficient}}="
            + f"{n}"
        ))


for control in (cheb_var, cheb_eps, cheb_delta):
    control.observe(update_cheb, names="value")

display(widgets.VBox([
    widgets.HBox([cheb_var, cheb_eps, cheb_delta]),
    cheb_output,
]))
update_cheb()


For

$$
\sigma^2=4,
\qquad
\varepsilon=0.05,
\qquad
\delta=0.01,
$$

Chebyshev gives the sufficient sample size

$$
\boxed{
n\ge160{,}000.
}
$$

The guarantee is rigorous for every $n$ but may be conservative.


### Monte Carlo central limit theorem

If

$$
0<\sigma^2<\infty,
$$

then

$$
\boxed{
\frac{
\sqrt n(
\widehat\theta_n-\theta
)
}{
\sigma
}
\xrightarrow{d}
N(0,1).
}
$$

For indicator payoffs, this is exactly the usual normal approximation to an empirical frequency.


## 5. Studentized Monte Carlo error

The variance is usually unknown.

Estimate it by

$$
\widehat\sigma_n^2
=
\frac1{n-1}
\sum_{i=1}^n
(Y_i-\overline Y_n)^2.
$$

Then

$$
\boxed{
\widehat{\operatorname{SE}}
(
\widehat\theta_n
)
=
\frac{\widehat\sigma_n}{\sqrt n}.
}
$$

Under the Monte Carlo CLT assumptions,

$$
\boxed{
\frac{
\sqrt n(
\widehat\theta_n-\theta
)
}{
\widehat\sigma_n
}
\xrightarrow{d}
N(0,1).
}
$$


### Asymptotic Monte Carlo interval

For large $n$, an approximate $100(1-\alpha)\%$ interval is

$$
\boxed{
\widehat\theta_n
\pm
z_{1-\alpha/2}
\frac{\widehat\sigma_n}{\sqrt n}.
}
$$

Its coverage converges to $1-\alpha$.

Except in special models, this is **not** an exact finite-sample interval.


In [ ]:
ci_N = widgets.IntSlider(
    value=5000,
    min=100,
    max=50000,
    step=100,
    description="N",
)
ci_alpha = widgets.FloatSlider(
    value=0.05,
    min=0.01,
    max=0.20,
    step=0.01,
    description="alpha",
)
ci_output = widgets.Output()


def update_mc_ci(*_):
    with ci_output:
        clear_output(wait=True)

        N = ci_N.value
        alpha = ci_alpha.value

        rng = np.random.default_rng(2026)
        u = rng.random(N)
        y = np.exp(u)

        lower, upper, estimate, se = asymptotic_ci(
            y,
            alpha=alpha,
        )

        display(Math(
            r"\widehat\theta="
            + f"{estimate:.8f}"
        ))
        display(Math(
            r"\widehat{\operatorname{SE}}="
            + f"{se:.8f}"
        ))
        display(Markdown(
            f"Approximate interval: **[{lower:.8f}, {upper:.8f}]**"
        ))
        display(Math(
            r"\theta=e-1="
            + f"{math.e-1:.8f}"
        ))


for control in (ci_N, ci_alpha):
    control.observe(update_mc_ci, names="value")

display(widgets.VBox([
    widgets.HBox([ci_N, ci_alpha]),
    ci_output,
]))
update_mc_ci()


### What the interval does not cover

A Monte Carlo interval quantifies random sampling error under the assumed simulation model.

It does **not** automatically account for:

- model misspecification;
- parameter uncertainty;
- programming errors;
- deterministic discretization bias;
- accidental dependence in the simulation algorithm.


## 6. Generating samples from a target distribution

A Monte Carlo estimator is only useful if we can generate observations from the required law.

The chapter develops two fundamental methods:

$$
\boxed{
\text{inverse transformation}
}
$$

and

$$
\boxed{
\text{rejection sampling}.
}
$$


### Inverse-transform sampling

For a cdf $F$, define the generalized inverse

$$
\boxed{
F^{-1}(u)
=
\inf\{x:F(x)\ge u\},
\qquad
0<u<1.
}
$$

If

$$
U\sim U(0,1),
$$

then

$$
\boxed{
X=F^{-1}(U)
}
$$

has cdf $F$.


### Exponential example

For

$$
X\sim\operatorname{Exp}(\lambda),
$$

$$
F^{-1}(u)
=
-\frac1\lambda
\log(1-u).
$$

Therefore

$$
\boxed{
X
=
-\frac1\lambda
\log(1-U)
}
$$

is exactly exponential.

Because $1-U$ is again uniform,

$$
-\lambda^{-1}\log U
$$

has the same law.


In [ ]:
inv_lam = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=5,
    step=0.1,
    description="lambda",
)
inv_N = widgets.IntSlider(
    value=20000,
    min=1000,
    max=100000,
    step=1000,
    description="N",
)
inv_output = widgets.Output()


def update_inverse_transform(*_):
    with inv_output:
        clear_output(wait=True)

        lam = inv_lam.value
        N = inv_N.value

        rng = np.random.default_rng(2026)
        u = rng.random(N)
        x = -np.log1p(-u)/lam

        display(Math(
            r"\overline X="
            + f"{x.mean():.6f}"
        ))
        display(Math(
            r"\mathbb E[X]="
            + f"{1/lam:.6f}"
        ))

        grid = np.linspace(0, np.quantile(x,0.995),500)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.hist(x,bins=60,density=True,alpha=0.35)
        ax.plot(grid,lam*np.exp(-lam*grid))
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.set_title("Inverse-transform exponential sampling")
        plt.show()


for control in (inv_lam, inv_N):
    control.observe(update_inverse_transform, names="value")

display(widgets.VBox([
    widgets.HBox([inv_lam, inv_N]),
    inv_output,
]))
update_inverse_transform()


### Why the generalized inverse matters

Suppose

$$
P(X=0)=\frac13,
$$

and conditional on $X>0$,

$$
X\sim U(0,1).
$$

Then one valid generalized inverse is

$$
F^{-1}(u)
=
\begin{cases}
0,
&0<u\le1/3,\\
\dfrac{3u-1}{2},
&1/3<u<1.
\end{cases}
$$

The cdf has a jump at zero, so an ordinary inverse function is not available globally.


In [ ]:
rng = np.random.default_rng(2026)
u = rng.random(30000)

x = np.where(
    u <= 1/3,
    0.0,
    (3*u-1)/2,
)

display(Math(
    r"\widehat P(X=0)="
    + f"{np.mean(x==0):.6f}"
))
display(Math(
    r"P(X=0)=\frac13"
))


## 7. Rejection sampling

Let $f$ be the target density and $q$ an easy proposal density.

Suppose

$$
\boxed{
f(x)\le Mq(x)
}
$$

almost everywhere for some finite $M\ge1$.

Generate independently

$$
X\sim q,
\qquad
U\sim U(0,1),
$$

and accept $X$ when

$$
\boxed{
U
\le
\frac{f(X)}{Mq(X)}.
}
$$

Then

$$
\boxed{
P(\text{accept})
=
\frac1M,
}
$$

and the accepted values have density $f$.


### Target $f(x)=2x$ on $[0,1]$

Use the uniform proposal

$$
q(x)=1,
\qquad
0\le x\le1.
$$

Since

$$
2x\le2q(x),
$$

take $M=2$.

The acceptance condition becomes

$$
U\le X.
$$

The mean acceptance rate is therefore

$$
\boxed{
1/2.
}
$$


In [ ]:
rej_N = widgets.IntSlider(
    value=30000,
    min=1000,
    max=100000,
    step=1000,
    description="proposals",
)
rej_output = widgets.Output()


def update_rejection(*_):
    with rej_output:
        clear_output(wait=True)

        N = rej_N.value
        accepted = rejection_beta21(N, seed=2026)

        display(Math(
            r"\widehat P(\text{accept})="
            + f"{accepted.size/N:.6f}"
        ))
        display(Math(
            r"\text{accepted sample mean}="
            + f"{accepted.mean():.6f}"
        ))
        display(Math(r"\mathbb E_f[X]=\frac23"))

        grid = np.linspace(0,1,500)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.hist(accepted,bins=50,density=True,alpha=0.35)
        ax.plot(grid,2*grid)
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.set_title("Rejection sampling from f(x)=2x")
        plt.show()


rej_N.observe(update_rejection, names="value")
display(widgets.VBox([rej_N, rej_output]))
update_rejection()


### Efficiency

Each proposal is accepted with probability $1/M$.

The number of independent proposals required to obtain one acceptance is geometric with mean

$$
\boxed{
M.
}
$$

Thus a proposal should make $M$ as small as practical while remaining easy to simulate.


## 8. Monte Carlo integration

If

$$
U\sim U(0,1)
$$

and $h$ is integrable,

$$
\boxed{
\int_0^1h(u)\,du
=
\mathbb E[h(U)].
}
$$

Thus

$$
\boxed{
\int_0^1h(u)\,du
\approx
\frac1n
\sum_{i=1}^nh(U_i).
}
$$

More generally, if $U$ is uniform on $[0,1]^d$,

$$
\int_{[0,1]^d}h(u)\,du
=
\mathbb E[h(U)].
$$


When

$$
h(U)\in L^2,
$$

plain Monte Carlo still has root-mean-square rate

$$
n^{-1/2}.
$$

The rate does not explicitly deteriorate with the dimension $d$, although the variance constant and the cost of evaluating $h$ can become much worse.


### Estimating $\pi$ by hit-or-miss integration

With uniform $(U,V)$ on the unit square,

$$
Y
=
4
\mathbf 1_{\{U^2+V^2\le1\}},
$$

and

$$
\mathbb E[Y]=\pi.
$$

The resulting estimator is numerical. It is not a proof of the mathematical value of $\pi$.


## 9. Bias, variance and mean squared error

Not every simulation estimator is unbiased.

Define

$$
\operatorname{Bias}(T_n)
=
\mathbb E[T_n]-\theta.
$$

Then

$$
\boxed{
\operatorname{MSE}(T_n;\theta)
=
\operatorname{Var}(T_n)
+
\operatorname{Bias}(T_n)^2.
}
$$

A small deterministic bias can sometimes reduce total mean squared error.


### A biased estimator with smaller MSE

Suppose $Y_1,\ldots,Y_{10}$ are i.i.d. with

$$
\theta=1,
\qquad
\operatorname{Var}(Y_i)=1.
$$

The unbiased estimator $\overline Y_{10}$ has

$$
\operatorname{MSE}=0.1.
$$

For

$$
T=0.9\overline Y_{10},
$$

$$
\operatorname{Bias}(T)=-0.1,
$$

$$
\operatorname{Var}(T)=0.081,
$$

so

$$
\boxed{
\operatorname{MSE}(T)=0.091<0.1.
}
$$


In [ ]:
shrink_a = widgets.FloatSlider(
    value=0.9,
    min=0.5,
    max=1.2,
    step=0.01,
    description="a",
)
shrink_output = widgets.Output()


def update_bias_variance(*_):
    with shrink_output:
        clear_output(wait=True)

        a = shrink_a.value

        # Mean theta=1, Var(Ybar_10)=0.1.
        bias = a-1
        variance = a*a*0.1
        mse = variance+bias*bias

        display(Math(r"\operatorname{Bias}(a\overline Y)=" + f"{bias:.6f}"))
        display(Math(r"\operatorname{Var}(a\overline Y)=" + f"{variance:.6f}"))
        display(Math(r"\operatorname{MSE}=" + f"{mse:.6f}"))


shrink_a.observe(update_bias_variance, names="value")
display(widgets.VBox([shrink_a, shrink_output]))
update_bias_variance()


### Sampling error versus deterministic approximation bias

If a model must first be discretized numerically, increasing the number of Monte Carlo replications can make sampling error arbitrarily small while leaving the discretization bias essentially unchanged.

These are different error sources and should be reported separately.


## 10. Antithetic variates

If

$$
U\sim U(0,1),
$$

then

$$
1-U\sim U(0,1).
$$

Instead of two independent evaluations, pair

$$
g(U)
$$

with

$$
g(1-U).
$$

Define

$$
\boxed{
A
=
\frac{
g(U)+g(1-U)
}{2}.
}
$$

Then

$$
\mathbb E[A]
=
\mathbb E[g(U)].
$$


### Why monotonicity helps

If $g$ is monotone and $g(U)\in L^2$, then

$$
\boxed{
\operatorname{Cov}
(
g(U),
g(1-U)
)
\le0.
}
$$

Hence

$$
\boxed{
\operatorname{Var}
\left(
\frac{g(U)+g(1-U)}2
\right)
\le
\frac12
\operatorname{Var}(g(U)).
}
$$

The right side is exactly the variance of the average of two independent evaluations.


### Exact example: $g(u)=u^2$

For $U\sim U(0,1)$,

$$
\operatorname{Var}(U^2)
=
\frac4{45}.
$$

The average of two independent evaluations has variance

$$
\boxed{
\frac2{45}.
}
$$

The antithetic pair has variance

$$
\boxed{
\frac1{180}.
}
$$

Thus the antithetic variance is eight times smaller.


In [ ]:
display(Math(r"\operatorname{Var}(U^2)=\frac4{45}"))
display(Math(r"\operatorname{Var}(\text{independent pair mean})=\frac2{45}"))
display(Math(r"\operatorname{Var}(\text{antithetic pair})=\frac1{180}"))
display(Math(
    r"\frac{(2/45)}{(1/180)}="
    + f"{(2/45)/(1/180):g}"
))


In [ ]:
anti_N = widgets.IntSlider(
    value=20000,
    min=1000,
    max=100000,
    step=1000,
    description="pairs",
)
anti_output = widgets.Output()


def update_antithetic(*_):
    with anti_output:
        clear_output(wait=True)

        N = anti_N.value
        rng = np.random.default_rng(2026)

        u1 = rng.random(N)
        u2 = rng.random(N)

        independent = 0.5*(u1*u1+u2*u2)

        u = rng.random(N)
        antithetic = 0.5*(u*u+(1-u)**2)

        display(Math(
            r"\widehat{\operatorname{Var}}(\text{independent pair})="
            + f"{independent.var(ddof=1):.8f}"
        ))
        display(Math(
            r"\widehat{\operatorname{Var}}(\text{antithetic pair})="
            + f"{antithetic.var(ddof=1):.8f}"
        ))


anti_N.observe(update_antithetic, names="value")
display(widgets.VBox([anti_N, anti_output]))
update_antithetic()


## 11. Control variates

Suppose $Y$ has unknown mean

$$
\theta=\mathbb E[Y],
$$

and $C$ is correlated with $Y$ but has known mean

$$
\mu_C=\mathbb E[C].
$$

For any constant $b$, define

$$
\boxed{
Z_b
=
Y-b(C-\mu_C).
}
$$

Then

$$
\mathbb E[Z_b]
=
\theta.
$$


### Optimal coefficient

If

$$
\operatorname{Var}(C)>0,
$$

then the variance-minimizing coefficient is

$$
\boxed{
b^*
=
\frac{
\operatorname{Cov}(Y,C)
}{
\operatorname{Var}(C)
}.
}
$$

The minimum variance is

$$
\boxed{
\operatorname{Var}(Z_{b^*})
=
\operatorname{Var}(Y)
-
\frac{
\operatorname{Cov}(Y,C)^2
}{
\operatorname{Var}(C)
}.
}
$$

When $\operatorname{Var}(Y)>0$,

$$
\boxed{
\operatorname{Var}(Z_{b^*})
=
\operatorname{Var}(Y)
(1-\rho_{Y,C}^2).
}
$$


In [ ]:
ctrl_rho = widgets.FloatSlider(
    value=0.9,
    min=-0.99,
    max=0.99,
    step=0.01,
    description="rho",
)
ctrl_output = widgets.Output()


def update_control_rho(*_):
    with ctrl_output:
        clear_output(wait=True)

        rho = ctrl_rho.value
        fraction = 1-rho*rho

        display(Math(
            r"\frac{\operatorname{Var}(Z_{b^*})}{\operatorname{Var}(Y)}="
            + f"{fraction:.6f}"
        ))
        display(Math(
            r"\text{variance removed fraction}="
            + f"{rho*rho:.6f}"
        ))


ctrl_rho.observe(update_control_rho, names="value")
display(widgets.VBox([ctrl_rho, ctrl_output]))
update_control_rho()


A control with correlation

$$
\rho=0.9
$$

leaves only

$$
1-0.9^2
=
0.19
$$

of the original single-observation variance.

It removes $81\%$ of the variance in the ideal optimal-coefficient calculation.


In [ ]:
# Control variate for integral_0^1 exp(u) du using C=U.
control_N = widgets.IntSlider(
    value=30000,
    min=2000,
    max=100000,
    step=1000,
    description="N",
)
control_output = widgets.Output()


def update_control_sim(*_):
    with control_output:
        clear_output(wait=True)

        N = control_N.value
        rng = np.random.default_rng(2026)

        pilot = 5000
        u_pilot = rng.random(pilot)
        y_pilot = np.exp(u_pilot)

        b = control_optimal_coefficient(
            y_pilot,
            u_pilot,
        )

        u = rng.random(N)
        y = np.exp(u)
        z = y-b*(u-0.5)

        plain_se = np.std(y,ddof=1)/math.sqrt(N)
        control_se = np.std(z,ddof=1)/math.sqrt(N)

        display(Math(r"\widehat b=" + f"{b:.6f}"))
        display(Math(r"\widehat{\operatorname{SE}}_{\mathrm{plain}}=" + f"{plain_se:.8f}"))
        display(Math(r"\widehat{\operatorname{SE}}_{\mathrm{control}}=" + f"{control_se:.8f}"))


control_N.observe(update_control_sim, names="value")
display(widgets.VBox([control_N, control_output]))
update_control_sim()


### Pilot-sample caution

If the same data are used both to estimate $b$ and to compute the final controlled mean, exact unbiasedness can be lost.

A clean unbiased workflow is:

1. estimate $b$ on an independent pilot sample;
2. hold that coefficient fixed;
3. use a fresh production sample for the final estimator.


## 12. Multivariate control variates

Let

$$
\mathbf C
=
(C_1,\ldots,C_m)^\top
$$

have known mean vector $\boldsymbol\mu_C$ and positive-definite covariance matrix $\Sigma_C$.

Define

$$
\mathbf c
=
\operatorname{Cov}(\mathbf C,Y).
$$

For

$$
Z_{\mathbf b}
=
Y
-
\mathbf b^\top
(
\mathbf C-\boldsymbol\mu_C
),
$$

the optimal coefficient is

$$
\boxed{
\mathbf b^*
=
\Sigma_C^{-1}
\mathbf c.
}
$$

The minimum variance is

$$
\boxed{
\operatorname{Var}(Y)
-
\mathbf c^\top
\Sigma_C^{-1}
\mathbf c.
}
$$


In [ ]:
Sigma = np.array([
    [2.0,1.0],
    [1.0,3.0],
])

c = np.array([1.0,2.0])

b_star = np.linalg.solve(Sigma,c)
reduction = float(c @ b_star)

display(Markdown(f"Optimal coefficient: **{b_star}**"))
display(Math(
    r"\mathbf c^\top\Sigma_C^{-1}\mathbf c="
    + f"{reduction:.6f}"
))


For the exercise

$$
\Sigma_C
=
\begin{pmatrix}
2&1\\
1&3
\end{pmatrix},
\qquad
\mathbf c
=
\begin{pmatrix}
1\\
2
\end{pmatrix},
$$

the optimal coefficient is

$$
\boxed{
\mathbf b^*
=
\begin{pmatrix}
1/5\\
3/5
\end{pmatrix},
}
$$

and the single-observation variance is reduced by

$$
\boxed{
7/5.
}
$$


## 13. Stratified sampling

Let $A_1,\ldots,A_K$ be a measurable partition with known probabilities

$$
p_k=P(A_k)>0.
$$

By total expectation,

$$
\theta
=
\sum_{k=1}^K
p_k
\mathbb E[Y\mid A_k].
$$

If $\overline Y_k$ is the sample mean from $n_k$ independent simulations inside stratum $k$, define

$$
\boxed{
\widehat\theta_{\mathrm{str}}
=
\sum_{k=1}^K
p_k
\overline Y_k.
}
$$


### Stratified variance

Let

$$
\sigma_k^2
=
\operatorname{Var}(Y\mid A_k).
$$

If simulations are independent across strata,

$$
\boxed{
\operatorname{Var}
(
\widehat\theta_{\mathrm{str}}
)
=
\sum_{k=1}^K
\frac{
p_k^2\sigma_k^2
}{
n_k
}.
}
$$


### Proportional allocation

If

$$
n_k=np_k
$$

are positive integers and

$$
\mathcal G
=
\sigma(A_1,\ldots,A_K),
$$

then

$$
\boxed{
\operatorname{Var}
(
\widehat\theta_{\mathrm{str}}
)
=
\frac1n
\mathbb E[
\operatorname{Var}(Y\mid\mathcal G)
].
}
$$

The law of total variance gives

$$
\boxed{
\operatorname{Var}
(
\widehat\theta_{\mathrm{plain}}
)
-
\operatorname{Var}
(
\widehat\theta_{\mathrm{str}}
)
=
\frac1n
\operatorname{Var}
(
\mathbb E[Y\mid\mathcal G]
).
}
$$

Thus proportional stratification removes the between-stratum component from the Monte Carlo error.


### Two-stratum example

Suppose both strata have probability $1/2$, both conditional variances equal $1$, and the conditional means are $0$ and $4$.

Then

$$
\operatorname{Var}(Y)=1+4=5.
$$

Plain Monte Carlo has variance

$$
\frac5n.
$$

Proportional stratification has variance

$$
\boxed{
\frac1n.
}
$$


In [ ]:
strat_between = widgets.FloatSlider(
    value=4.0,
    min=0,
    max=10,
    step=0.25,
    description="mean gap",
)
strat_output = widgets.Output()


def update_stratification_gap(*_):
    with strat_output:
        clear_output(wait=True)

        gap = strat_between.value

        # Two equal strata, conditional variances 1 each,
        # means 0 and gap.
        between_var = gap*gap/4
        total_var = 1+between_var

        display(Math(
            r"\operatorname{Var}(Y)="
            + f"{total_var:.6f}"
        ))
        display(Math(r"n\operatorname{Var}(\widehat\theta_{\mathrm{str}})=1"))
        display(Math(
            r"\text{variance reduction numerator}="
            + f"{between_var:.6f}"
        ))


strat_between.observe(update_stratification_gap, names="value")
display(widgets.VBox([strat_between, strat_output]))
update_stratification_gap()


## 14. Neyman allocation

If stratum standard deviations differ and simulation cost per observation is equal, the optimal continuous allocation is

$$
\boxed{
n_k^*
=
n
\frac{
p_k\sigma_k
}{
\sum_jp_j\sigma_j
}.
}
$$

The minimized continuous-relaxation variance is

$$
\boxed{
\frac1n
\left(
\sum_kp_k\sigma_k
\right)^2.
}
$$

More effort is sent to strata that are large and internally volatile.


### Example

Let

$$
p_1=0.8,
\qquad
p_2=0.2,
$$

and

$$
\sigma_1=1,
\qquad
\sigma_2=4.
$$

Proportional allocation has variance

$$
\frac4n.
$$

Neyman allocation gives

$$
n_1^*
=
n_2^*
=
\frac n2,
$$

with variance

$$
\boxed{
\frac{2.56}{n}.
}
$$


In [ ]:
neyman_p1 = widgets.FloatSlider(
    value=0.8,
    min=0.05,
    max=0.95,
    step=0.05,
    description="p1",
)
neyman_s1 = widgets.FloatSlider(
    value=1,
    min=0.1,
    max=6,
    step=0.1,
    description="sigma1",
)
neyman_s2 = widgets.FloatSlider(
    value=4,
    min=0.1,
    max=6,
    step=0.1,
    description="sigma2",
)
neyman_output = widgets.Output()


def update_neyman(*_):
    with neyman_output:
        clear_output(wait=True)

        p1 = neyman_p1.value
        p2 = 1-p1
        s1 = neyman_s1.value
        s2 = neyman_s2.value

        denom = p1*s1+p2*s2

        alloc1 = p1*s1/denom
        alloc2 = p2*s2/denom

        prop_constant = p1*s1*s1+p2*s2*s2
        neyman_constant = denom*denom

        display(Math(
            r"\frac{n_1^*}{n}="
            + f"{alloc1:.6f}"
        ))
        display(Math(
            r"\frac{n_2^*}{n}="
            + f"{alloc2:.6f}"
        ))
        display(Math(
            r"nV_{\mathrm{proportional}}="
            + f"{prop_constant:.6f}"
        ))
        display(Math(
            r"nV_{\mathrm{Neyman}}="
            + f"{neyman_constant:.6f}"
        ))


for control in (neyman_p1,neyman_s1,neyman_s2):
    control.observe(update_neyman, names="value")

display(widgets.VBox([
    widgets.HBox([neyman_p1, neyman_s1, neyman_s2]),
    neyman_output,
]))
update_neyman()


## 15. Importance sampling as change of measure

Let $P\ll Q$ and define the Radon--Nikodym derivative

$$
L
=
\frac{dP}{dQ}.
$$

For an integrable payoff $h$,

$$
\boxed{
\mathbb E_P[h]
=
\mathbb E_Q[hL].
}
$$

For densities,

$$
\theta
=
\int h(x)f(x)\,dx.
$$

If $q$ is a proposal density, define

$$
w(x)
=
\frac{f(x)}{q(x)}
$$

on $\{q>0\}$.

Then

$$
\boxed{
\theta
=
\mathbb E_q[
h(X)w(X)
].
}
$$


### Support condition

The proposal must satisfy

$$
\boxed{
|h(x)|f(x)=0
}
$$

for almost every $x$ where

$$
q(x)=0.
$$

If the proposal gives zero density on a genuine part of the target integrand, the change-of-measure identity fails.

Insufficient support is not merely inefficient. It changes the target.


### Importance-sampling estimator

For independent

$$
X_i\sim q,
$$

define

$$
\boxed{
\widehat\theta_n^{\mathrm{IS}}
=
\frac1n
\sum_{i=1}^n
h(X_i)
w(X_i).
}
$$

It is unbiased under the support and integrability conditions.

If the second moment is finite,

$$
\boxed{
\operatorname{Var}
(
\widehat\theta_n^{\mathrm{IS}}
)
=
\frac1n
\left[
\int_{\{q>0\}}
h(x)^2
\frac{f(x)^2}{q(x)}
\,dx
-
\theta^2
\right].
}
$$


### Formal zero-variance proposal

For $h\ge0$ and $0<\theta<\infty$, the formal ideal proposal is

$$
\boxed{
q^*(x)
=
\frac{
h(x)f(x)
}{
\theta
}.
}
$$

Then the weighted payoff is identically $\theta$.

The proposal is usually unavailable because it contains the unknown target itself, but it reveals the design principle: sample where $h(x)f(x)$ is large.


## 16. Rare normal tail by importance sampling

Let

$$
Z\sim N(0,1),
$$

and target

$$
\theta=P(Z>4).
$$

Then

$$
\theta
\approx
3.167\times10^{-5}.
$$

Plain Monte Carlo sees the event only about once in thirty thousand draws.


Instead, sample from

$$
X\sim N(4,1).
$$

If $f$ is the standard normal density and $q$ the $N(4,1)$ density, then

$$
\boxed{
\frac{f(x)}{q(x)}
=
e^{8-4x}.
}
$$

The estimator is

$$
\boxed{
\widehat\theta_n^{\mathrm{IS}}
=
\frac1n
\sum_{i=1}^n
\mathbf 1_{\{X_i>4\}}
e^{8-4X_i}.
}
$$

Under the proposal,

$$
P_q(X>4)=1/2.
$$


In [ ]:
is_N = widgets.IntSlider(
    value=50000,
    min=5000,
    max=300000,
    step=5000,
    description="N",
)
is_seed = widgets.IntSlider(
    value=2026,
    min=0,
    max=5000,
    description="seed",
)
is_output = widgets.Output()


def update_importance_tail(*_):
    with is_output:
        clear_output(wait=True)

        N = is_N.value
        seed = is_seed.value
        rng = np.random.default_rng(seed)

        z = rng.standard_normal(N)
        plain = (z>4).astype(float)

        x = 4+rng.standard_normal(N)
        is_payoff = normal_tail_is_payoff(
            x,
            shift=4,
        )

        exact = normal_sf(4)

        display(Math(r"\theta=" + f"{exact:.10g}"))
        display(Math(
            r"\widehat\theta_{\mathrm{plain}}="
            + f"{plain.mean():.10g}"
        ))
        display(Math(
            r"\widehat{\operatorname{SE}}_{\mathrm{plain}}="
            + f"{plain.std(ddof=1)/math.sqrt(N):.10g}"
        ))
        display(Math(
            r"\widehat\theta_{\mathrm{IS}}="
            + f"{is_payoff.mean():.10g}"
        ))
        display(Math(
            r"\widehat{\operatorname{SE}}_{\mathrm{IS}}="
            + f"{is_payoff.std(ddof=1)/math.sqrt(N):.10g}"
        ))


for control in (is_N, is_seed):
    control.observe(update_importance_tail, names="value")

display(widgets.VBox([
    widgets.HBox([is_N, is_seed]),
    is_output,
]))
update_importance_tail()


For this simple shift proposal, the chapter gives an importance-sampling single-draw variance of approximately

$$
4.525\times10^{-9},
$$

versus a plain single-draw variance of approximately

$$
3.167\times10^{-5}.
$$

The variance reduction factor is about

$$
\boxed{
7.0\times10^3.
}
$$


In [ ]:
theta = normal_sf(4)
plain_var = theta*(1-theta)

is_second = math.exp(16)*normal_sf(8)
is_var = is_second-theta*theta

display(Math(r"\theta=" + f"{theta:.12g}"))
display(Math(r"V_{\mathrm{plain}}=" + f"{plain_var:.12g}"))
display(Math(r"V_{\mathrm{IS}}=" + f"{is_var:.12g}"))
display(Math(
    r"\frac{V_{\mathrm{plain}}}{V_{\mathrm{IS}}}="
    + f"{plain_var/is_var:.4f}"
))


## 17. Rare-event estimation and relative error

For

$$
\theta=P(X\in A)
$$

with a very small $\theta$, plain Monte Carlo uses an indicator payoff.

Its exact relative standard error is

$$
\boxed{
\frac{
\sqrt{
\operatorname{Var}(\widehat\theta_n)
}
}{
\theta
}
=
\sqrt{
\frac{
1-\theta
}{
n\theta
}
}.
}
$$

For $\theta\ll1$,

$$
\boxed{
\text{relative SE}
\approx
\frac1{\sqrt{n\theta}}.
}
$$


To target relative standard error $r$ when $\theta$ is small,

$$
\boxed{
n
\approx
\frac1{\theta r^2}.
}
$$

For

$$
\theta=10^{-6},
\qquad
r=0.1,
$$

plain Monte Carlo needs roughly

$$
\boxed{
10^8
}
$$

replications.


In [ ]:
rare_theta = widgets.FloatLogSlider(
    value=1e-6,
    base=10,
    min=-9,
    max=-2,
    step=0.25,
    description="theta",
)
rare_r = widgets.FloatSlider(
    value=0.10,
    min=0.02,
    max=0.50,
    step=0.01,
    description="rel SE",
)
rare_output = widgets.Output()


def update_rare_sample_size(*_):
    with rare_output:
        clear_output(wait=True)

        theta = rare_theta.value
        r = rare_r.value

        n = 1/(theta*r*r)

        display(Math(
            r"n\approx"
            + f"{n:.6g}"
        ))
        display(Math(
            r"\mathbb E[\text{rare hits}]\approx n\theta="
            + f"{n*theta:.6g}"
        ))


for control in (rare_theta, rare_r):
    control.observe(update_rare_sample_size, names="value")

display(widgets.VBox([
    widgets.HBox([rare_theta, rare_r]),
    rare_output,
]))
update_rare_sample_size()


### One hundred expected rare events

If

$$
\theta=10^{-8},
$$

and we want the expected number of observed events to equal $100$, then

$$
n\theta=100.
$$

Thus

$$
\boxed{
n=10^{10}.
}
$$

This is a direct illustration of why rare-event simulation often needs targeted variance reduction.


## 18. Historical problem: from physical chance to electronic Monte Carlo

Buffon's needle used random experiments to evaluate mathematical quantities long before electronic computers.

The modern computational Monte Carlo method was formalized in the late 1940s. The chapter connects the Los Alamos development associated with Stanislaw Ulam, John von Neumann and Nicholas Metropolis with the broader idea of using probability as a computational tool.


### From Buffon's experiment to a computer integral

Consider

$$
I
=
\int_0^1
4\sqrt{1-u^2}\,du
=
\pi.
$$

With

$$
U_i\stackrel{\mathrm{i.i.d.}}{\sim}U(0,1),
$$

define

$$
\boxed{
\widehat I_n
=
\frac1n
\sum_{i=1}^n
4\sqrt{1-U_i^2}.
}
$$

This estimator is unbiased for $\pi$.


The second moment is

$$
\mathbb E[
16(1-U^2)
]
=
\frac{32}{3}.
$$

Hence

$$
\boxed{
\operatorname{Var}
(
4\sqrt{1-U^2}
)
=
\frac{32}{3}
-
\pi^2.
}
$$

Therefore

$$
\boxed{
\operatorname{Var}
(
\widehat I_n
)
=
\frac{
32/3-\pi^2
}{
n
}.
}
$$


Chebyshev gives the sufficient condition

$$
\frac{
32/3-\pi^2
}{
n(0.01)^2
}
\le
0.05.
$$

Thus

$$
\boxed{
n\ge159{,}413
}
$$

is sufficient for

$$
P(
|\widehat I_n-\pi|
<
0.01
)
\ge
0.95.
$$


In [ ]:
historical_n = math.ceil(
    (32/3-math.pi**2)
    /
    (0.05*(0.01**2))
)

display(Math(
    r"n_{\mathrm{sufficient}}="
    + f"{historical_n}"
))


### Direct integration versus hit-or-miss

Let

$$
H
=
4
\mathbf 1_{\{U^2+V^2\le1\}},
$$

and

$$
D
=
4\sqrt{1-U^2}.
$$

Then

$$
\operatorname{Var}(H)
=
4\pi-\pi^2,
$$

while

$$
\operatorname{Var}(D)
=
\frac{32}{3}-\pi^2.
$$

Moreover,

$$
\boxed{
D
=
\mathbb E[H\mid U].
}
$$

The law of total variance explains the reduction: conditioning removes the extra binary simulation noise from $V$ while preserving the mean.


In [ ]:
var_hit = 4*math.pi-math.pi**2
var_direct = 32/3-math.pi**2

display(Math(r"\operatorname{Var}(H)=" + f"{var_hit:.8f}"))
display(Math(r"\operatorname{Var}(D)=" + f"{var_direct:.8f}"))
display(Math(
    r"\frac{\operatorname{Var}(H)}{\operatorname{Var}(D)}="
    + f"{var_hit/var_direct:.6f}"
))


In [ ]:
compare_N = widgets.IntSlider(
    value=30000,
    min=1000,
    max=100000,
    step=1000,
    description="N",
)
compare_output = widgets.Output()


def update_pi_estimators(*_):
    with compare_output:
        clear_output(wait=True)

        N = compare_N.value
        rng = np.random.default_rng(2026)

        u = rng.random(N)
        v = rng.random(N)

        h = hit_miss_pi_payoff(u,v)
        d = direct_pi_payoff(u)

        display(Math(
            r"\widehat{\operatorname{SE}}_{\mathrm{hit}}="
            + f"{h.std(ddof=1)/math.sqrt(N):.8f}"
        ))
        display(Math(
            r"\widehat{\operatorname{SE}}_{\mathrm{direct}}="
            + f"{d.std(ddof=1)/math.sqrt(N):.8f}"
        ))


compare_N.observe(update_pi_estimators, names="value")
display(widgets.VBox([compare_N, compare_output]))
update_pi_estimators()


## 19. Random-number generation and reproducibility

A computer normally uses a pseudorandom number generator.

A reproducible simulation report should record, when material:

- generator and software version;
- seed or seed sequence;
- model parameters;
- number of replications;
- variance-reduction scheme;
- exact estimator;
- uncertainty calculation.

A library call does not by itself prove the exact independence assumptions of the mathematical theorem.


In [ ]:
rng1 = np.random.default_rng(2026)
rng2 = np.random.default_rng(2026)

a = rng1.random(5)
b = rng2.random(5)

display(Markdown(f"First stream: **{a}**"))
display(Markdown(f"Second stream: **{b}**"))
display(Markdown(f"Same seed reproduces the stream: **{np.allclose(a,b)}**"))


## 20. Monte Carlo error versus model risk

Increasing $n$ reduces random sampling noise around the **model-implied** target.

It does not force the model itself to become realistic.

A simulation may have a tiny reported standard error and still be practically unreliable because of:

- model misspecification;
- structural change;
- omitted state variables;
- parameter uncertainty;
- discretization choices;
- programming errors.

Monte Carlo precision and model validity are different layers of uncertainty.


## 21. Python laboratory: plain Monte Carlo and variance reduction

We compare four estimators of

$$
\int_0^1e^x\,dx
=
e-1:
$$

1. plain Monte Carlo;
2. antithetic variates;
3. a control variate $C=U$ with known mean $1/2$;
4. two-stratum sampling.

The control coefficient is estimated on an independent pilot sample and then held fixed.


In [ ]:
lab_N = widgets.IntSlider(
    value=40000,
    min=4000,
    max=200000,
    step=2000,
    description="budget",
)
lab_seed = widgets.IntSlider(
    value=2026,
    min=0,
    max=5000,
    description="seed",
)
lab_output = widgets.Output()


def update_full_lab(*_):
    with lab_output:
        clear_output(wait=True)

        N = lab_N.value
        seed = lab_seed.value
        rng = np.random.default_rng(seed)

        exact = math.e-1

        # Plain MC with N evaluations.
        u = rng.random(N)
        y = np.exp(u)

        plain_est = y.mean()
        plain_se = y.std(ddof=1)/math.sqrt(N)

        # Antithetic uses N evaluations = N/2 pairs.
        m = N//2
        u = rng.random(m)
        pair = 0.5*(np.exp(u)+np.exp(1-u))

        anti_est = pair.mean()
        anti_se = pair.std(ddof=1)/math.sqrt(m)

        # Pilot coefficient independent of production run.
        pilot_size = min(5000,max(500,N//10))
        up = rng.random(pilot_size)
        yp = np.exp(up)

        b = control_optimal_coefficient(
            yp,
            up,
        )

        u = rng.random(N)
        y = np.exp(u)
        z = y-b*(u-0.5)

        control_est = z.mean()
        control_se = z.std(ddof=1)/math.sqrt(N)

        # Two strata, N total approximately split evenly.
        m1 = N//2
        m2 = N-m1

        u1 = 0.5*rng.random(m1)
        u2 = 0.5+0.5*rng.random(m2)

        y1 = np.exp(u1)
        y2 = np.exp(u2)

        strat_est = 0.5*y1.mean()+0.5*y2.mean()
        strat_var = (
            0.25*y1.var(ddof=1)/m1
            +
            0.25*y2.var(ddof=1)/m2
        )
        strat_se = math.sqrt(strat_var)

        rows = [
            "| estimator | estimate | estimated SE | absolute error |",
            "|---|---:|---:|---:|",
            f"| plain | {plain_est:.8f} | {plain_se:.8f} | {abs(plain_est-exact):.8f} |",
            f"| antithetic | {anti_est:.8f} | {anti_se:.8f} | {abs(anti_est-exact):.8f} |",
            f"| control | {control_est:.8f} | {control_se:.8f} | {abs(control_est-exact):.8f} |",
            f"| stratified | {strat_est:.8f} | {strat_se:.8f} | {abs(strat_est-exact):.8f} |",
        ]

        display(Markdown("\n".join(rows)))
        display(Math(r"e-1=" + f"{exact:.8f}"))
        display(Math(r"\widehat b_{\mathrm{pilot}}=" + f"{b:.6f}"))


for control in (lab_N,lab_seed):
    control.observe(update_full_lab, names="value")

display(widgets.VBox([
    widgets.HBox([lab_N,lab_seed]),
    lab_output,
]))
update_full_lab()


## 22. A reliable Monte Carlo workflow

A mathematically defensible simulation study should make the following structure explicit.

1. **Target:** write the quantity as an expectation, probability or integral.
2. **Model:** state the distribution under which the target is defined.
3. **Estimator:** write the exact random estimator.
4. **Sampling law:** state how observations are generated and verify support or acceptance conditions.
5. **Bias:** identify analytic or discretization bias.
6. **Variance:** derive it when possible or estimate it from the run.
7. **Error statement:** distinguish finite-sample guarantees from asymptotic CLT intervals.
8. **Diagnostics:** check sensitivity to sample size, seeds and algorithm choices.
9. **Model risk:** distinguish within-model Monte Carlo uncertainty from uncertainty about the model.


## 23. Solved-style checks


### Sample size from known variance

If

$$
\operatorname{Var}(Y)=25,
$$

then

$$
\operatorname{SE}(\widehat\theta_n)
=
\frac5{\sqrt n}.
$$

To make this at most $0.01$,

$$
\boxed{
n\ge250{,}000.
}
$$


### Chebyshev guarantee

If

$$
\operatorname{Var}(Y)=4,
$$

and we require

$$
P(
|\widehat\theta_n-\theta|
\ge0.05
)
\le0.01,
$$

then

$$
\boxed{
n\ge160{,}000.
}
$$


### Control-variate reduction

If

$$
\operatorname{Var}(Y)=9,
\qquad
\operatorname{Var}(C)=4,
\qquad
\operatorname{Cov}(Y,C)=3,
$$

then

$$
\boxed{
b^*=\frac34,
}
$$

and

$$
\boxed{
\operatorname{Var}(Z_{b^*})
=
\frac{27}{4}.
}
$$


In [ ]:
b_star = 3/4
min_var = 9-3**2/4

display(Math(r"b^*=" + f"{b_star:.6f}"))
display(Math(r"V_{\min}=" + f"{min_var:.6f}"))


### Importance-sampling identity

For the exponential target density

$$
f(x)
=
e^{-x}
\mathbf 1_{\{x\ge0\}},
$$

and target

$$
\theta=P_f(X>a),
$$

any proposal $q$ that is positive on $(a,\infty)$ gives the unbiased estimator

$$
\boxed{
\widehat\theta_n^{\mathrm{IS}}
=
\frac1n
\sum_{i=1}^n
\mathbf 1_{\{X_i>a\}}
\frac{
e^{-X_i}
}{
q(X_i)
},
\qquad
X_i\sim q.
}
$$


## 24. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Plain MC","plain"),
        ("Chebyshev","cheb"),
        ("Inverse transform","inverse"),
        ("Rejection","rejection"),
        ("Bias--variance","bias"),
        ("Antithetic","antithetic"),
        ("Control","control"),
        ("Stratification","strat"),
        ("Importance","importance"),
        ("Rare event","rare"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "plain",
            "cheb",
            "inverse",
            "rejection",
            "bias",
            "antithetic",
            "control",
            "strat",
            "importance",
            "rare",
        ])

    if kind == "plain":
        target = "100"
        prompt = "By what factor must n increase to reduce plain Monte Carlo SE by a factor of 10?"
        hint = "SE is proportional to n^{-1/2}."
        solution = r"\text{Increase }n\text{ by a factor of }100."

    elif kind == "cheb":
        target = "160000"
        prompt = "Var(Y)=4, epsilon=0.05, delta=0.01. Give the Chebyshev sufficient n."
        hint = "Use variance/(delta epsilon^2)."
        solution = r"n\ge160000."

    elif kind == "inverse":
        target = "0.5"
        prompt = "For X~Exp(2), what is E[X] after inverse-transform sampling?"
        hint = "The simulation method changes how X is generated, not its law."
        solution = r"\mathbb E[X]=1/2."

    elif kind == "rejection":
        target = "0.5"
        prompt = "A rejection sampler uses M=2. What is its acceptance probability?"
        hint = "Acceptance probability is 1/M."
        solution = r"P(\text{accept})=1/2."

    elif kind == "bias":
        target = "0.091"
        prompt = "If Bias(T)=-0.1 and Var(T)=0.081, find MSE(T)."
        hint = "MSE=variance+bias^2."
        solution = r"\operatorname{MSE}=0.091."

    elif kind == "antithetic":
        target = "8"
        prompt = "For g(u)=u^2, by what factor is the independent-pair variance 2/45 larger than the antithetic variance 1/180?"
        hint = "Divide the two variances."
        solution = r"\text{factor}=8."

    elif kind == "control":
        target = "0.75"
        prompt = "Var(C)=4 and Cov(Y,C)=3. Find the optimal one-dimensional control coefficient."
        hint = "Use Cov(Y,C)/Var(C)."
        solution = r"b^*=3/4."

    elif kind == "strat":
        target = "2.56"
        prompt = "For p=(0.8,0.2), sigma=(1,4), what is n times the Neyman-allocation variance?"
        hint = "Square sum p_k sigma_k."
        solution = r"nV_{\min}=2.56."

    elif kind == "importance":
        target = "no"
        prompt = "Can q(x)=0 on a positive-measure region where |h(x)|f(x)>0 and still give the correct basic IS identity? yes/no"
        hint = "Check the support condition."
        solution = r"\text{No.}"

    else:
        target = "10000000000"
        prompt = "If theta=1e-8, how many plain samples give 100 expected rare hits?"
        hint = "Solve n theta=100."
        solution = r"n=10^{10}."

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Write the exact estimator and its variance or support condition first.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 25. AI Audit: Monte Carlo claims

Use this checklist on any AI-generated Monte Carlo argument.

1. Is the target written explicitly as an expectation, probability or integral?
2. Is the probability model under which the target is defined stated?
3. Is the exact random estimator written down?
4. Is unbiasedness claimed only after computing its expectation?
5. Are MSE, RMSE and standard error distinguished?
6. Is the $n^{-1/2}$ square-root law applied correctly?
7. Is consistency being confused with a finite-sample error guarantee?
8. Is a Chebyshev statement labeled as rigorous but potentially conservative?
9. Is a CLT interval described as asymptotic rather than exact?
10. Is the sample variance used in studentization with the required finite nonzero variance assumption?
11. Is a tiny Monte Carlo SE incorrectly interpreted as evidence that the real-world model is correct?
12. Does inverse-transform sampling use the generalized inverse of the cdf?
13. Does a rejection sampler verify $f\le Mq$ almost everywhere?
14. Is low rejection acceptance being confused with bias in the accepted sample?
15. Is rejection efficiency recognized as an issue of $M$, not correctness of accepted draws?
16. Is Monte Carlo integration expressed as an expectation under the stated uniform law?
17. Is deterministic numerical bias separated from sampling variance?
18. Is MSE decomposed as variance plus squared bias?
19. Is antithetic variance reduction justified through covariance rather than visual symmetry alone?
20. Is the monotonicity condition checked before applying the antithetic guarantee?
21. Does a control variate have a known expectation?
22. Is the optimal coefficient $b^*=\operatorname{Cov}(Y,C)/\operatorname{Var}(C)$?
23. If the same data estimate $b$ and the final mean, is possible loss of exact unbiasedness acknowledged?
24. For multiple controls, is the covariance matrix positive definite before inversion?
25. Is the stratified estimator weighted by the true stratum probabilities?
26. Is its variance computed as $\sum p_k^2\sigma_k^2/n_k$?
27. Is proportional stratification connected correctly to the law of total variance?
28. Is Neyman allocation proportional to $p_k\sigma_k$?
29. Does importance sampling check its support condition before forming $f/q$?
30. Is the importance weight target density divided by proposal density, not the reverse?
31. Is importance sampling being called “always better” without a variance calculation?
32. Is the rare-event relative error recognized as roughly $1/\sqrt{n\theta}$?
33. Are pseudorandom seed and algorithm choices recorded for reproducibility?
34. Is simulation used as numerical evidence rather than proof of an analytic identity?
35. Is model risk kept separate from Monte Carlo sampling error?

### Claims to audit

- “If the reported Monte Carlo standard error is tiny, the model prediction is reliable in the real world.”
- “Importance sampling is always more accurate than plain Monte Carlo because it uses a smarter proposal.”
- “If a rejection sampler accepts very rarely, its accepted observations are biased.”
- “A $95\%$ CLT interval has exact $95\%$ coverage for every finite sample size.”

All four claims are false as written.


### Correct replacements

A small Monte Carlo standard error means the estimator has small estimated sampling variability **under the chosen model and algorithm**. It says nothing by itself about model misspecification.

Importance sampling can reduce variance dramatically, but a poor proposal can increase variance or even make the relevant second moment infinite.

A rejection sampler with a valid envelope produces the exact target conditional law among accepted proposals. A low acceptance rate means inefficiency, not bias.

A studentized CLT interval has **asymptotic** coverage under the theorem's assumptions; exact finite-sample coverage is not generally guaranteed.


### Suggested AI-guided activities

- “Give me one target expectation and require me to construct a plain estimator, antithetic estimator, control-variate estimator and importance-sampling estimator. Make me prove unbiasedness before discussing code.”
- “Give me a rare-event probability, make me estimate the plain-Monte-Carlo relative error, then design a proposal and check its support.”
- “Give me a rejection sampler and make me derive the acceptance probability and expected number of proposals per accepted observation.”
- “Give me a two-stratum problem and make me compare proportional and Neyman allocation.”
- “Generate a plausible Monte Carlo report with one model-risk mistake and one variance-reduction mistake, then make me audit it.”


## 26. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Plain Monte Carlo SE decreases at rate:",
        ["Choose...","1/n","1/sqrt(n)","1/log(n)"],
        "1/sqrt(n)",
        r"\operatorname{SE}(\widehat\theta_n)=\sigma/\sqrt n.",
    ),
    (
        "2. Strong Monte Carlo consistency requires:",
        ["Choose...","E|Y|<infinity","finite fourth moment","normal Y"],
        "E|Y|<infinity",
        r"\mathbb E|Y|<\infty\text{ is sufficient under iid sampling.}",
    ),
    (
        "3. A Chebyshev Monte Carlo error guarantee is:",
        ["Choose...","finite-sample","only asymptotic"],
        "finite-sample",
        r"P(|\widehat\theta_n-\theta|\ge\varepsilon)\le\sigma^2/(n\varepsilon^2).",
    ),
    (
        "4. A standard studentized Monte Carlo CLT interval is exact for every n:",
        ["Choose...","true","false"],
        "false",
        r"\text{Its general justification is asymptotic.}",
    ),
    (
        "5. Inverse-transform sampling uses:",
        ["Choose...","the generalized inverse cdf","the density derivative only"],
        "the generalized inverse cdf",
        r"X=F^{-1}(U).",
    ),
    (
        "6. Rejection sampling with envelope constant M accepts with probability:",
        ["Choose...","1/M","M","1-M"],
        "1/M",
        r"P(\text{accept})=1/M.",
    ),
    (
        "7. MSE equals:",
        ["Choose...","variance plus bias squared","variance plus bias"],
        "variance plus bias squared",
        r"\operatorname{MSE}=\operatorname{Var}+\operatorname{Bias}^2.",
    ),
    (
        "8. For a monotone g, antithetic pairing guarantees covariance:",
        ["Choose...","nonpositive","strictly positive","zero always"],
        "nonpositive",
        r"\operatorname{Cov}(g(U),g(1-U))\le0.",
    ),
    (
        "9. The optimal one-dimensional control coefficient is:",
        [
            "Choose...",
            "Cov(Y,C)/Var(C)",
            "Var(Y)/Cov(Y,C)",
            "Cov(Y,C)/Var(Y)",
        ],
        "Cov(Y,C)/Var(C)",
        r"b^*=\operatorname{Cov}(Y,C)/\operatorname{Var}(C).",
    ),
    (
        "10. Neyman allocation is proportional to:",
        ["Choose...","p_k sigma_k","p_k/sigma_k","sigma_k only"],
        "p_k sigma_k",
        r"n_k^*\propto p_k\sigma_k.",
    ),
    (
        "11. Importance weights use:",
        ["Choose...","target/proposal","proposal/target"],
        "target/proposal",
        r"w=f/q.",
    ),
    (
        "12. Tiny Monte Carlo SE proves the model is correct:",
        ["Choose...","true","false"],
        "false",
        r"\text{Sampling error and model risk are distinct.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )

    quiz_widgets.append(dropdown)

    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:700px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(
            f"### Score: {score}/{len(quiz_data)}"
        ))

        for i, (
            widget,
            (_,_,correct,explanation),
        ) in enumerate(zip(quiz_widgets,quiz_data),1):

            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))

            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows+[grade_button,quiz_output]
))


## 27. Automatic mathematical verification

The final code cell checks representative formulas from every major method in the chapter.


In [ ]:
# Plain Monte Carlo square-root law.
sigma = 2.0
assert abs(sigma/math.sqrt(10_000)-0.02) < 1e-12
assert abs(sigma/math.sqrt(1_000_000)-0.002) < 1e-12

# Chebyshev sample size.
assert chebyshev_sample_size(4,0.05,0.01) == 160_000

# Known-variance SE requirement.
assert math.ceil((5/0.01)**2) == 250_000

# Bias--variance example.
mse_unbiased = 0.1
mse_shrink = 0.9**2*0.1 + (-0.1)**2
assert abs(mse_shrink-0.091) < 1e-12
assert mse_shrink < mse_unbiased

# Antithetic exact variances.
var_u2 = 1/5-(1/3)**2
ind_pair_var = var_u2/2
anti_var = 1/180

assert abs(var_u2-4/45) < 1e-12
assert abs(ind_pair_var-2/45) < 1e-12
assert abs(ind_pair_var/anti_var-8) < 1e-12

# One-dimensional control.
b_star = 3/4
min_var = 9-3**2/4

assert abs(b_star-0.75) < 1e-12
assert abs(min_var-27/4) < 1e-12

# Multivariate control exercise.
Sigma = np.array([
    [2.0,1.0],
    [1.0,3.0],
])
c = np.array([1.0,2.0])

b = np.linalg.solve(Sigma,c)
reduction = float(c@b)

assert np.allclose(b,[1/5,3/5])
assert abs(reduction-7/5) < 1e-12

# Proportional versus Neyman example.
p1,p2 = 0.8,0.2
s1,s2 = 1.0,4.0

prop_constant = p1*s1*s1+p2*s2*s2
neyman_constant = (p1*s1+p2*s2)**2

assert abs(prop_constant-4) < 1e-12
assert abs(neyman_constant-2.56) < 1e-12

# Rare-event relative-error rule.
theta = 1e-6
r = 0.1
assert abs(1/(theta*r*r)-1e8) < 1e-6

# Expected rare hits exercise.
assert abs(100/1e-8-1e10) < 1e-6

# Historical pi estimator variance and sample size.
var_direct = 32/3-math.pi**2
historical_n = math.ceil(var_direct/(0.05*0.01**2))

assert historical_n == 159_413

# Hit-or-miss pi variance is larger.
var_hit = 4*math.pi-math.pi**2
assert var_direct < var_hit

# Rare normal tail IS variance.
theta = normal_sf(4)
plain_var = theta*(1-theta)
is_second = math.exp(16)*normal_sf(8)
is_var = is_second-theta*theta

assert is_var > 0
assert plain_var/is_var > 6000

# Rejection sampler theoretical target moments.
# f(x)=2x on [0,1]: E[X]=2/3.
assert abs(2/3-0.6666666666666666) < 1e-12

# Generalized inverse atom construction.
def mixed_quantile(u):
    if u <= 1/3:
        return 0.0
    return (3*u-1)/2

assert mixed_quantile(0.2) == 0
assert 0 < mixed_quantile(0.8) < 1

show_result(
    "All Chapter 15 automatic checks passed",
    r"\operatorname{SE}(\widehat\theta_n)=\sigma/\sqrt n",
    r"\operatorname{MSE}=\operatorname{Var}+\operatorname{Bias}^2",
    r"b^*=\frac{\operatorname{Cov}(Y,C)}{\operatorname{Var}(C)}",
    r"n_k^*\propto p_k\sigma_k",
    r"\mathbb E_P[h]=\mathbb E_Q[h\,dP/dQ]",
    note=(
        "Square-root law, Chebyshev, antithetic, control-variate, stratification, "
        "rare-event and historical-pi checks all passed."
    ),
)


## 28. Chapter map

| Chapter concept | Computational representation |
|---|---|
| plain Monte Carlo | empirical mean of simulated payoffs |
| MSE/RMSE/SE | explicit error measures |
| square-root law | log--log $n^{-1/2}$ graph |
| strong consistency | running hit-or-miss $\pi$ estimate |
| Chebyshev guarantee | sample-size calculator |
| Monte Carlo CLT | asymptotic standardized error |
| studentization | estimated SE and interval |
| inverse transform | exponential and mixed atom/continuous example |
| rejection sampling | target $f(x)=2x$ |
| rejection efficiency | geometric mean number of proposals |
| Monte Carlo integration | expectation under uniform sampling |
| bias--variance decomposition | shrinkage example |
| antithetic variates | exact $g(u)=u^2$ variance comparison |
| one-dimensional control | optimal $b^*$ and correlation reduction |
| pilot control | independent pilot coefficient |
| multivariate controls | covariance-matrix solution |
| stratification | conditional means and variances |
| proportional allocation | total-variance interpretation |
| Neyman allocation | optimal $p_k\sigma_k$ allocation |
| importance sampling | change-of-measure identity |
| support condition | target contribution must be covered |
| zero-variance principle | formal optimal proposal |
| rare normal tail | shifted-normal importance sampling |
| rare-event relative error | $1/\sqrt{n\theta}$ rule |
| historical Monte Carlo | direct integral estimator of $\pi$ |
| conditional variance idea | direct $\pi$ estimator versus hit-or-miss |
| reproducibility | seed and generator record |
| model risk | within-model error versus model uncertainty |
| reliable workflow | nine-step simulation checklist |
| AI Audit | correctness and uncertainty checks |

The central computational principle is:

$$
\boxed{
\text{do not merely simulate more}
\quad
\text{when you can simulate more intelligently}.
}
$$

Variance reduction changes the variance constant while preserving the target, whereas the square-root law governs how plain independent replication scales with sample size.
